# **GraL Data Processing**
---
Gaia GraL X catalogue (Ducourant et al. 2026) \
VizieR: https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/707/A345 \
Paper: https://doi.org/10.1051/0004-6361/202558049

**Cuts:** quads with exactly one lens galaxy (`G*`), exactly four images (`A*`–`F`), and positions for every image and the galaxy.

In [30]:
#> imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#> file imports
from astropy.table import Table

#> module imports
sys.path.insert(1, '../../../opus/')
import plot
import lensing
import astrometry
import modules.geometry as geometry

### ***Reading the Catalogue***
---
`hdu=1`: components (table A1) &emsp; `hdu=2`: time delays (table B1)

In [66]:
#> catalogue file
file = 'gaia_full_table.fit'

#> component & time-delay tables
df  = Table.read(file, hdu=1).to_pandas()
tds = Table.read(file, hdu=2).to_pandas()

#> bytes --> strings
for table in [df, tds]:
    for key in table.columns[ table.dtypes == object ]:
        table[key] = table[key].str.decode('utf-8').str.strip()

#> quads only
df = df[ df['Type'] == 'Quad' ].sort_values(['Name', 'Comp']).reset_index(drop=True)
df[ df['Name'] == 'WISE025942.9-163543' ]

,TimeDelay,Name,Comp,RAJ2000,DEJ2000,Type,MaxSep,rpos,BibCode,RApdeg,...,zSHSRC,GLens,CompId,RACodeg,e_RACodeg,DECodeg,e_DECodeg,GaiaDR3,AllWISE,recno
369,*,WISE025942.9-163543,A,44.928792,-16.595361,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928792,...,NaN,NaN,0,NaN,NaN,NaN,NaN,5153828508862119040,NaN,153
370,*,WISE025942.9-163543,B,44.928700,-16.595117,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928699,...,NaN,NaN,0,NaN,NaN,NaN,NaN,5153828504567283072,NaN,152
371,*,WISE025942.9-163543,C,44.928365,-16.595205,Quad,1.575,Gaia_DR3,2018RNAAS...2...21S,44.928363,...,NaN,NaN,0,NaN,NaN,NaN,NaN,5153828508862118912,NaN,149
372,*,WISE025942.9-163543,D,44.928584,-16.595469,Quad,1.575,HST,2018RNAAS...2...21S,44.928584,...,NaN,NaN,0,NaN,NaN,NaN,NaN,-9223372036854775808,NaN,150
373,*,WISE025942.9-163543,G,44.928591,-16.595277,Quad,1.575,HST,2018RNAAS...2...21S,44.928590,...,NaN,NaN,0,NaN,NaN,NaN,NaN,-9223372036854775808,NaN,151


### ***Checking Publication Values***
---
Components whose best astrometry is the publication, but whose best and publication coordinates differ.

In [32]:
#> best astrometry from publication
pub = df[ df['rpos'] == 'Publication' ].copy()

#> best - publication  [deg]
pub['RA_diff'] = pub['RAJ2000'] - pub['RApdeg']
pub['DE_diff'] = pub['DEJ2000'] - pub['DEpdeg']

#> incorrectly copied
issues = pub[ (pub['RA_diff'].abs() >= 1E-6) | (pub['DE_diff'].abs() >= 1E-6) ]
issues[['Name', 'Comp', 'RA_diff', 'DE_diff']]

,Name,Comp,RA_diff,DE_diff
44,B1555+375,A,0.000021,-0.000018
45,B1555+375,B,0.000037,-0.000005
46,B1555+375,C,0.000001,0.000002
47,B1555+375,D,0.000007,-0.000004
48,B1555+375,G,0.000066,-0.000017
49,B1608+656,A,0.000126,-0.000032
271,PSJ042913.17+142840.9,B,0.000119,0.000155
334,SDSSJ2222+2745,A,0.000021,0.000010


### ***Lens Summary***
---
Number of galaxies (`G*`) and images (`A*`–`F`) per lens.

In [33]:
#> number of each component per lens
counts = pd.crosstab(df['Name'], df['Comp'])

#> summary
lenses = pd.DataFrame({'n_defs': counts.filter(regex='^G').sum(axis=1),
                       'n_ims' : counts.filter(regex='^[A-F]').sum(axis=1)})
lenses = lenses.join(df.groupby('Name')[['zdef', 'zsource']].first())

#> configurations
lenses.value_counts(['n_defs', 'n_ims']).sort_index()

n_defs  n_ims
0       0         8
        2         1
        3         3
        4         6
1       3         2
        4        49
        6         1
2       4         5
        5         1
3       4         1
        6         2
5       4         1
Name: count, dtype: int64

### ***One-Galaxy Quads***
---
Exactly one galaxy, exactly four images, and no missing positions.

In [34]:
#> exactly one galaxy & four images
names = lenses[ (lenses['n_defs'] == 1) & (lenses['n_ims'] == 4) ].index
quads = df[ df['Name'].isin(names) & df['Comp'].str.match('^[A-G]') ].copy()

#> adopted positions: best astrometry  [deg]
quads['ra']  = quads['RAJ2000']
quads['dec'] = quads['DEJ2000']

#> publication values (best astrometry does not match publication)
use_pub = quads['Name'].isin(['HE1113-0641', 'SDSS1138+0314'])
quads.loc[use_pub, ['ra', 'dec']] = quads.loc[use_pub, ['RApdeg', 'DEpdeg']].to_numpy()

#> SDSS1138+0314 A: publication does not match SIMBAD --> SIMBAD  [deg]
quads.loc[ (quads['Name'] == 'SDSS1138+0314') & (quads['Comp'] == 'A'), ['ra', 'dec'] ] = [174.515542, 3.249389]

#> dropping lenses with any missing position
missing = quads.loc[ quads[['ra', 'dec']].isna().any(axis=1), 'Name' ].unique()
quads   = quads[ ~quads['Name'].isin(missing) ].reset_index(drop=True)
print(f'> {quads["Name"].nunique()} one-galaxy quads (dropped for missing positions: {list(missing)})')

> 49 one-galaxy quads (dropped for missing positions: [])


In [35]:
#> galaxy position for each component's lens  [deg]
gal = quads[ quads['Comp'].str.startswith('G') ].set_index('Name')
gra = gal.loc[ quads['Name'], 'ra' ].to_numpy()
gde = gal.loc[ quads['Name'], 'dec'].to_numpy()

#> offsets from the galaxy  [arcsec]
quads['dRA'], quads['dDEC'] = astrometry.RADEC_to_ARCSEC(quads['ra'].to_numpy(), quads['dec'].to_numpy(), gra, gde)
quads[['Name', 'Comp', 'rpos', 'dRA', 'dDEC']]

,Name,Comp,rpos,dRA,dDEC
0,2MASSJ11344050-2103230,A,Gaia_DR3,0.752181,-0.779
1,2MASSJ11344050-2103230,B,Gaia_DR3,1.481036,0.976
2,2MASSJ11344050-2103230,C,Gaia_DR3,-1.195462,-1.552
3,2MASSJ11344050-2103230,D,Gaia_DR3,-0.495545,0.587
4,2MASSJ11344050-2103230,G,HST,0.000000,0.000
...,...,...,...,...,...
240,WISE025942.9-163543,A,Gaia_DR3,0.693842,-0.301
241,WISE025942.9-163543,B,Gaia_DR3,0.376630,0.576
242,WISE025942.9-163543,C,Gaia_DR3,-0.778177,0.261
243,WISE025942.9-163543,D,HST,-0.023000,-0.691


### ***Einstein Radius Cut***
---
Einstein radius approximated by the mean distance of the four images from the galaxy; keeping $\theta_E < 2.5''$.

In [36]:
#> Einstein radius: mean image distance from galaxy  [arcsec]
ims    = quads[ ~quads['Comp'].str.startswith('G') ]
thetaE = np.hypot(ims['dRA'], ims['dDEC']).groupby(ims['Name']).mean()

#> keeping thetaE < 2.5 arcsec
removed = thetaE[ thetaE >= 2.5 ].round(2).to_dict()
quads   = quads[ quads['Name'].isin(thetaE[ thetaE < 2.5 ].index) ].reset_index(drop=True)
print(f'> {quads["Name"].nunique()} quads with thetaE < 2.5 arcsec (removed: {removed})')

> 47 quads with thetaE < 2.5 arcsec (removed: {'GraLJ081828.3-261325': 2.99, 'SDSS1004+4112': 8.02})


### ***Arrival Order & Observables***
---

In [37]:
#> lensing observables
observables = ['t12', 't23', 't34', 'd2/d1', 'd3/d1', 'd4/d1', 'dt23', 'd01', 'd02', 'd03', 'd04']

#> iterating through each lens
rows = []
for name, lens in quads.groupby('Name'):

    #> images relative to galaxy  [arcsec]
    ims    = lens[ ~lens['Comp'].str.startswith('G') ]
    images = ims[['dRA', 'dDEC']].to_numpy()
    labels = ims['Comp'].to_numpy()

    #> morphological arrival order
    ordered = lensing.arrivalOrder(images, 0.0, 0.0)
    order   = ''.join(labels[ np.where((images == im).all(axis=1))[0][0] ] for im in ordered)

    #> observables (lensObs expects a 5th image row --> NaN placeholder)
    lobs = geometry.lensObs((0, 0), np.vstack([ordered, [np.nan, np.nan]]), observables)
    rows.append([name, order] + [float(x) for x in lobs])

obs = pd.DataFrame(rows, columns=['Name', 'mo_order'] + observables)
obs

,Name,mo_order,t12,t23,t34,d2/d1,d3/d1,d4/d1,dt23,d01,d02,d03,d04
0,2MASSJ11344050-2103230,CBAD,160.990859,79.388255,176.174471,0.905398,0.552758,0.392132,1.421837,1.959039,1.773709,1.082875,0.768202
1,2MASXJ01471020+4630433,CBAD,67.496249,33.282576,176.414406,1.053678,0.974821,0.551357,4.304327,2.131754,2.246182,2.078077,1.175357
2,ATLASJ2344-3056,BDCA,162.337535,78.863321,177.402246,1.037732,0.986751,0.773852,-0.623941,0.528391,0.548328,0.521390,0.408896
3,B0712+472,DABC,79.388131,11.777773,161.516727,0.998735,0.994495,0.575153,-4.345365,0.821000,0.819962,0.816480,0.472200
4,B1422+231,CABD,75.739283,29.040993,180.458461,0.930297,0.895670,0.245843,-9.364461,1.086350,1.010628,0.973011,0.267072
5,B1555+375,CABD,84.086977,16.737991,158.935955,0.888588,0.926361,0.330377,1.005672,0.327466,0.290982,0.303351,0.108187
6,B1608+656,BACD,115.410051,37.399853,163.846015,0.720885,0.802417,0.760000,-3.608825,1.318184,0.950258,1.057732,1.001820
7,COSMOS5921+0638,CABD,105.996296,42.394842,164.428572,0.977629,0.994530,0.944665,6.551909,0.730169,0.713834,0.726175,0.689765
8,DESJ0029-3814,CABD,179.533147,51.227799,107.062330,0.650409,0.423661,0.426003,-1.753763,1.339179,0.871015,0.567359,0.570494
9,DESJ0053-2012,ACBD,110.275839,33.431070,146.156380,1.028528,0.874074,0.334988,14.647397,1.839933,1.892422,1.608237,0.616356


### ***Comparing to Accepted Lenses***
---

In [38]:
#> accepted lenses
accepted = set(pd.read_csv('../quad_names.csv')['Name'].astype(str).str.strip())
gral     = set(obs['Name'])

#> comparison
print(f'> GraL one-galaxy quads: {len(gral)} | accepted: {len(accepted)} | both: {len(gral & accepted)}')
print(f'> GraL only:     {sorted(gral - accepted)}')
print(f'> accepted only: {sorted(accepted - gral)}')

> GraL one-galaxy quads: 47 | accepted: 62 | both: 45
> GraL only:     ['B1608+656', 'MGJ0414+0534']
> accepted only: ['1810.01633', 'B0128+437', 'B1933+503 A', 'B1933+503 B', 'HS0810+2554', 'HST14176+5226', 'J095930.94+023', 'J100140.12+020', 'LSDQ0047-2808', 'MG2016+112', 'SDSS1011+0143', 'SDSS1402+6321', 'SDSSJ1538+5817', 'SLACSJ0946+1006', 'SLACSJ1205+491', 'SLACSJ1636+470', 'SLACSJ2300+002']


### ***Saving Data to Files***
---
`quads`: one gal, four image systems and their info &emsp; `obs`: morphological arrival order and lensing observables

In [47]:
#> saving dataframes
quads.to_csv('2026GraL_quads.csv', index=False)
obs.to_csv('2026GraL_obs.csv', index=False)